# 3. Modeling

In [1]:
import pandas as pd

import numpy as np

from sklearn.model_selection import train_test_split, KFold, cross_val_score

from sklearn.pipeline import Pipeline

from sklearn.compose import ColumnTransformer

from sklearn.impute import SimpleImputer

from sklearn.preprocessing import OneHotEncoder

from sklearn.tree import DecisionTreeRegressor

In [2]:
RMSE_base = 0.21

## 3.1. Оптимизация препроцессора

**Загрузим данные для обучения**

In [3]:
df = pd.read_csv('E:/ML/housing-prices-ml/data/raw/train.csv')
y = np.log1p(df['SalePrice'])
X = df.drop(columns=['SalePrice', 'Id'])

**Разделим данные на обучающую и тестовыую выборки**

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y, 
    train_size=0.8,
    shuffle=True,
    random_state=46)

Разделим признаки на числовые и катгориальные для корректного построения пайплайна обучения


In [5]:
num_features = X.select_dtypes('number').columns.to_list()
cat_features = X.select_dtypes('str').columns.to_list()

Преобразования по результатам EDA

In [6]:
num_features.remove('MSSubClass')
cat_features.append('MSSubClass')

**Рассмотрим влияние гипотез, выдвинутных на этапе EDA**

In [7]:
absence_features = [
    'PoolQC',
    'FireplaceQu',
    'GarageQual',
    'GarageCond',
    'GarageFinish',
    'GarageType',
    'BsmtQual',
    'BsmtCond',
    'BsmtExposure',
    'BsmtFinType1',
    'BsmtFinType2',
    'Alley',
    'Fence',
    'MiscFeature',
    'MasVnrType'
]

regular_cat_features = []

for feature in cat_features:
    if feature not in absence_features:
        regular_cat_features.append(feature)

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

absence_pipline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='None')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

regular_cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('cat_absence', absence_pipline, absence_features),
    ('cat_reg', regular_cat_pipeline, regular_cat_features),
    ('num', num_pipeline, num_features)
])


Проверим что будет, если числовые признаки с малым количеством числовых значений представить категориальными

In [8]:
for feature in ['MoSold', 'YrSold', 'BsmtHalfBath', 'HalfBath', 'BsmtFullBath', 'FullBath', 'Fireplaces', 'KitchenAbvGr', 'GarageCars', 'BedroomAbvGr', 'TotRmsAbvGrd']:
    for i in range(2):
        test_num_features = num_features.copy()
        test_regular_cat_features = regular_cat_features.copy()
        test_absence_features = absence_features.copy()

        test_num_features.remove(feature)
        if i == 0:
            test_regular_cat_features.append(feature)
        else:
            test_absence_features.append(feature)

        preprocessor = ColumnTransformer([
        ('cat_absence', absence_pipline, absence_features),
        ('cat_reg', regular_cat_pipeline, test_regular_cat_features),
        ('num', num_pipeline, test_num_features)
        ])

        model = Pipeline([
        ('preprocessor', preprocessor),
        ('model', DecisionTreeRegressor(random_state=46))
        ])

        cv = KFold(
        n_splits=5,
        shuffle=True,
        random_state=46
        )

        cv_scores = cross_val_score(
            model,
            X_train,
            y_train,
            cv=cv,
            scoring='neg_root_mean_squared_error'
        )

        cv_rmse = -cv_scores

        if i == 0:
            print(f'Test: {feature} to regular_cat_features')
        else:
            print(f'Test: {feature} to absence_features')

        print(f'CV RMSE: {cv_rmse.mean():.2f} ± {cv_rmse.std():.2f}\n')

Test: MoSold to regular_cat_features
CV RMSE: 0.21 ± 0.01

Test: MoSold to absence_features
CV RMSE: 0.20 ± 0.01

Test: YrSold to regular_cat_features
CV RMSE: 0.20 ± 0.01

Test: YrSold to absence_features
CV RMSE: 0.21 ± 0.01

Test: BsmtHalfBath to regular_cat_features
CV RMSE: 0.21 ± 0.01

Test: BsmtHalfBath to absence_features
CV RMSE: 0.20 ± 0.01

Test: HalfBath to regular_cat_features
CV RMSE: 0.21 ± 0.01

Test: HalfBath to absence_features
CV RMSE: 0.21 ± 0.02

Test: BsmtFullBath to regular_cat_features
CV RMSE: 0.21 ± 0.02

Test: BsmtFullBath to absence_features
CV RMSE: 0.20 ± 0.01

Test: FullBath to regular_cat_features
CV RMSE: 0.20 ± 0.01

Test: FullBath to absence_features
CV RMSE: 0.20 ± 0.02

Test: Fireplaces to regular_cat_features
CV RMSE: 0.20 ± 0.01

Test: Fireplaces to absence_features
CV RMSE: 0.21 ± 0.02

Test: KitchenAbvGr to regular_cat_features
CV RMSE: 0.21 ± 0.02

Test: KitchenAbvGr to absence_features
CV RMSE: 0.21 ± 0.01

Test: GarageCars to regular_cat_feat

Данные монипуляции не улучшили качество

Сильная линейная связь, замеченная у некоторых числовых признаков не должна оказывать влияния на качество модели, основанных на деревьях

Рассмотрим попмжет ли повысить качество удаление сильно несбалансированных категориальных признаков

In [9]:
for feature in ['Street', 'Alley', 'Utilities', 'LandSlope',
                'Condition2', 'RoofMatl', 'Heating', 'CentralAir',
                'Electrical', 'Functional', 'GarageCond', 'PoolQC',
                'MiscFeature', 'Neighborhood', 'Condition1', 'Condition2',
                'HouseStyle', 'Exterior1st', 'Exterior2nd', 'SaleType']:
    
    test_num_features = num_features.copy()
    test_regular_cat_features = regular_cat_features.copy()
    test_absence_features = absence_features.copy()

    if feature in test_regular_cat_features:
        test_regular_cat_features.remove(feature)
    if feature in test_regular_cat_features:
        test_regular_cat_features.remove(feature)

    preprocessor = ColumnTransformer([
    ('cat_absence', absence_pipline, absence_features),
    ('cat_reg', regular_cat_pipeline, test_regular_cat_features),
    ('num', num_pipeline, test_num_features)
    ])

    model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', DecisionTreeRegressor(random_state=46))
    ])

    cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=46
    )

    cv_scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring='neg_root_mean_squared_error'
    )

    cv_rmse = -cv_scores
    
    print(f'Test: REMOVE {feature}')
    print(f'CV RMSE: {cv_rmse.mean():.2f} ± {cv_rmse.std():.2f}\n')

Test: REMOVE Street
CV RMSE: 0.21 ± 0.02

Test: REMOVE Alley
CV RMSE: 0.20 ± 0.02

Test: REMOVE Utilities
CV RMSE: 0.21 ± 0.02

Test: REMOVE LandSlope
CV RMSE: 0.20 ± 0.02

Test: REMOVE Condition2
CV RMSE: 0.21 ± 0.01

Test: REMOVE RoofMatl
CV RMSE: 0.21 ± 0.02

Test: REMOVE Heating
CV RMSE: 0.21 ± 0.02

Test: REMOVE CentralAir
CV RMSE: 0.21 ± 0.02

Test: REMOVE Electrical
CV RMSE: 0.20 ± 0.02

Test: REMOVE Functional
CV RMSE: 0.20 ± 0.02

Test: REMOVE GarageCond
CV RMSE: 0.20 ± 0.02

Test: REMOVE PoolQC
CV RMSE: 0.20 ± 0.02

Test: REMOVE MiscFeature
CV RMSE: 0.20 ± 0.02

Test: REMOVE Neighborhood
CV RMSE: 0.21 ± 0.01

Test: REMOVE Condition1
CV RMSE: 0.21 ± 0.01

Test: REMOVE Condition2
CV RMSE: 0.21 ± 0.01

Test: REMOVE HouseStyle
CV RMSE: 0.20 ± 0.02

Test: REMOVE Exterior1st
CV RMSE: 0.20 ± 0.01

Test: REMOVE Exterior2nd
CV RMSE: 0.20 ± 0.01

Test: REMOVE SaleType
CV RMSE: 0.20 ± 0.01



Данные монипуляции не улучшили качество

## Рассмотрим другие модели

Для анализа рассмотрим следуюшие модели с подбором гиперпараметров:
* Decision Tree 
* Random Forest
* Gradient Boosting
* XGBoost  

Подбор гиперпараметров осуществим методом случайного поиска по сетке

In [16]:
from sklearn.model_selection import RandomizedSearchCV

Зафиксируем предобработку

In [21]:
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

absence_pipline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='None')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

regular_cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('cat_absence', absence_pipline, absence_features),
    ('cat_reg', regular_cat_pipeline, regular_cat_features),
    ('num', num_pipeline, num_features)
])

**Произведем подбор гиперпараметров для решающего дерева**

In [26]:
model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', DecisionTreeRegressor(random_state=46))
    ])

param_grid = {
    "model__max_depth": [3, 5, 7, 10, 15, 20, 30, None],
    "model__min_samples_split": [2, 5, 10, 20],
    "model__min_samples_leaf": [1, 2, 4, 8, 12],
    "model__max_features": [1.0, "sqrt", "log2", 0.5, 0.75],
}

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=46
)

search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_grid,
    n_iter=100,
    cv=cv,
    scoring='neg_root_mean_squared_error',
    random_state=46,
    n_jobs=-1
)

search.fit(
    X_train,
    y_train
)

print("Best parameters:")
for param, value in search.best_params_.items():
    print(f"  {param}: {value}")

print(
    f"CV RMSE: {-search.best_score_:.2f} ± "
    f"{search.cv_results_['std_test_score'][search.best_index_]:.2f}"
)

Best parameters:
  model__min_samples_split: 10
  model__min_samples_leaf: 12
  model__max_features: 0.75
  model__max_depth: 15
CV RMSE: 0.19 ± 0.00


Подбор гиперпараметров дает результаты котрые лучше бэйзланйа

**Произведем подбор гиперпараметров для случайного леса**

In [28]:
from sklearn.ensemble import RandomForestRegressor

Best parameters:
  model__min_samples_split: 5
  model__min_samples_leaf: 1
  model__max_features: 0.75
  model__max_depth: None
CV RMSE: 0.15 ± 0.01


In [ ]:

model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=46))
    ])

param_grid = {
    "model__max_depth": [3, 5, 7, 10, 15, 20, 30, None],
    "model__min_samples_split": [2, 5, 10, 20],
    "model__min_samples_leaf": [1, 2, 4, 8, 12],
    "model__max_features": [1.0, "sqrt", "log2", 0.5, 0.75],
}

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=46
)

search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_grid,
    n_iter=100,
    cv=cv,
    scoring='neg_root_mean_squared_error',
    random_state=46,
    n_jobs=-1
)

search.fit(
    X_train,
    y_train
)

print("Best parameters:")
for param, value in search.best_params_.items():
    print(f"  {param}: {value}")

print(
    f"CV RMSE: {-search.best_score_:.2f} ± "
    f"{search.cv_results_['std_test_score'][search.best_index_]:.2f}"
)

Результаты улучшились

**Произведем подбор гиперпараметров для градиентного бустинга**

In [30]:
from sklearn.ensemble import GradientBoostingRegressor

In [31]:
model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', GradientBoostingRegressor(random_state=46))
    ])

param_grid = {
    "model__n_estimators": [100, 200, 300, 500, 700, 1000],
    "model__learning_rate": [0.01, 0.03, 0.05, 0.1, 0.15, 0.2],
    "model__max_depth": [2, 3, 4, 5, 6, 8],
    "model__min_samples_split": [2, 5, 10, 15, 20],
    "model__min_samples_leaf": [1, 2, 4, 8, 12],
    "model__subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
}

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=46
)

search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_grid,
    n_iter=100,
    cv=cv,
    scoring='neg_root_mean_squared_error',
    random_state=46,
    n_jobs=-1
)

search.fit(
    X_train,
    y_train
)

print("Best parameters:")
for param, value in search.best_params_.items():
    print(f"  {param}: {value}")

print(
    f"CV RMSE: {-search.best_score_:.2f} ± "
    f"{search.cv_results_['std_test_score'][search.best_index_]:.2f}"
)

Best parameters:
  model__subsample: 0.6
  model__n_estimators: 700
  model__min_samples_split: 2
  model__min_samples_leaf: 2
  model__max_depth: 2
  model__learning_rate: 0.1
CV RMSE: 0.13 ± 0.01


**Произведем подбор гиперпараметров для XGBoost**

In [32]:
from xgboost import XGBRFRegressor

In [34]:
model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', XGBRFRegressor(random_state=46))
    ])

param_grid = {
    "model__n_estimators": [200, 400, 600, 800, 1000],
    "model__learning_rate": [0.01, 0.03, 0.05, 0.1, 0.15],
    "model__max_depth": [2, 3, 4, 5, 6, 8],
    "model__min_child_weight": [1, 3, 5, 7, 10],
    "model__subsample": [0.7, 0.8, 0.9, 1.0],
    "model__colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
    "model__gamma": [0, 0.01, 0.1, 0.5, 1, 2, 5],
    "model__reg_alpha": [0, 0.001, 0.01, 0.1, 0.5, 1],
    "model__reg_lambda": [0.1, 0.5, 1, 2, 5, 10],
}

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=46
)

search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_grid,
    n_iter=1000,
    cv=cv,
    scoring='neg_root_mean_squared_error',
    random_state=46,
    n_jobs=-1
)

search.fit(
    X_train,
    y_train
)

print("Best parameters:")
for param, value in search.best_params_.items():
    print(f"  {param}: {value}")

print(
    f"CV RMSE: {-search.best_score_:.2f} ± "
    f"{search.cv_results_['std_test_score'][search.best_index_]:.2f}"
)

Best parameters:
  model__subsample: 0.8
  model__reg_lambda: 0.1
  model__reg_alpha: 0
  model__n_estimators: 600
  model__min_child_weight: 1
  model__max_depth: 6
  model__learning_rate: 0.15
  model__gamma: 0.1
  model__colsample_bytree: 0.7
CV RMSE: 0.35 ± 0.02


## Итоговое предсказание

Загрузим данные для обучения и прогноза

In [112]:
test_df = pd.read_csv("E:/ML/housing-prices-ml/data/raw/test.csv")
train_df = pd.read_csv("E:/ML/housing-prices-ml/data/raw/train.csv")

In [113]:
#y_train = np.log1p(train_df['SalePrice'])
y_train = train_df['SalePrice']
X_train = train_df.drop(columns=['SalePrice', 'Id'])

In [114]:
X_test = test_df

In [115]:
num_features = X.select_dtypes('number').columns.to_list()
cat_features = X.select_dtypes('str').columns.to_list()

num_features.remove('MSSubClass')

cat_features.append('MSSubClass')

absence_features = [
    'PoolQC',
    'FireplaceQu',
    'GarageQual',
    'GarageCond',
    'GarageFinish',
    'GarageType',
    'BsmtQual',
    'BsmtCond',
    'BsmtExposure',
    'BsmtFinType1',
    'BsmtFinType2',
    'Alley',
    'Fence',
    'MiscFeature',
    'MasVnrType'
]

regular_cat_features = []

for feature in cat_features:
    if feature not in absence_features:
        regular_cat_features.append(feature)

In [116]:
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

absence_pipline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='None')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

regular_cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('cat_absence', absence_pipline, absence_features),
    ('cat_reg', regular_cat_pipeline, regular_cat_features),
    ('num', num_pipeline, num_features)
])

In [117]:
model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', GradientBoostingRegressor(
        min_samples_split=5,
        min_samples_leaf=1,
        max_features=0.75,
        max_depth=None,
        random_state=46))
    ])

In [118]:
model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](79,)","['MSSubClass','MSZoning','LotFrontage',...,'YrSold','SaleType', 'SaleCondition']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,79
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat_absence', ...), ('cat_reg', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed throu

In [119]:
predict = model.predict(X_test)

In [120]:
result = pd.DataFrame(index=X_test['Id'])
result['SalePrice'] = predict
result

,SalePrice
Id,
1461,127291.824593
1462,155525.370621
1463,186662.345615
1464,183952.625614
1465,200491.739517
...,...
2915,79752.259830
2916,91289.863159
2917,162304.303715


In [121]:
result.to_csv("E:/ML/housing-prices-ml/data/result/new_submission.csv")